# Predicción desde química medida durante actividad — LAB019–021

**Pregunta:** si inicializamos el modelo durante una fermentación que ya emite CO₂ sostenido, ¿predice la química y el CO₂ posteriores **hasta el final**, incluyendo todas las nutriciones futuras?

Esta es la primera prueba retrospectiva **sin EKF, sin añadir latencia y sin reajustar parámetros**. No es desplazar una curva calculada ni demostrar que un detector puede inicializar automáticamente los estados. El reloj sigue indicando horas desde la inoculación; cada inicio también tiene su coordenada local τ=t−tₐ. Las muestras iniciales originales permanecen como información previa.

Se conservan los **parámetros antiguos del notebook 02**, y se compara theta_full con theta incompleto + seis defaults. Se utiliza la estructura simplificada ya empleada en `co2_local_observability`: transitorio dentro de producción biológica, sin término respiratorio separado; se mantienen O₂, rampa/gain, gate y pool. Los resultados están condicionados a esa estructura.

**Límite de los datos:** la muestra 7 (46 h) tiene N, GF, E y glicerol en los tres labs, pero carece de G/F válidos por separado. Se usa el último reparto G/F medido anterior y no marcado, declarando esa hipótesis y perturbándola. La muestra 10 (70,5 h) tiene química completa; es posterior al pulso en LAB019 y LAB021, y anterior en LAB020. No se borra ni se vuelve a aplicar un pulso pasado.

La biomasa Oculyze reciente es un **centro exploratorio**, no una medición fiable de biomasa activa ni un canal de validación. El CO₂ disuelto se inicializa por inversión de la ecuación de salida con los últimos intervalos completos de caudal. Ambos se someten a sensibilidad. **Ninguna banda de escenarios representa un intervalo de confianza.**

## 1. Configuración, parámetros e integridad
La preparación de datos siguiente reproduce de forma visible la auditoría del notebook de observabilidad: exports Y15, Excel offline, diluciones de etanol, Oculyze y logger. No se cargan trayectorias cacheadas como si fueran nuevas simulaciones. Se registran hashes de datos, calibraciones y código protegido.

In [ ]:
from __future__ import annotations

import csv
import io
import json
import math
import re
import sys
from dataclasses import replace
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
from IPython import get_ipython
_ip = get_ipython()
if _ip is not None:
    _ip.run_line_magic("matplotlib", "inline")
import numpy as np
import openpyxl
import pandas as pd
from IPython.display import Markdown, Image, display

ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "fermentation_model").exists())
FM = ROOT / "fermentation_model"
LAB2026 = FM / "laboratory_2026"
DATA_DIR = FM / "data" / "mem2026" / "LAB019-021"
DATA_016_018_DIR = FM / "data" / "mem2026" / "LAB016-018"
REFERENCE_RESULTS_DIR = LAB2026 / "results" / "lab019_021_theta_full_holdout"

for path in (FM, LAB2026, FM / "pilot_2025"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from shared import run_new_must_glycerol_estimability_doe as base
from shared import run_secondary_joint_campaign_doe as joint
from laboratory_2026 import run_estimability_historical_by_medium as natural_loader
from laboratory_2026 import run_co2_matrix_cross_validation_2026 as co2_cross

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.dpi": 110, "font.size": 9.5, "font.family": "DejaVu Sans"})


def display_inline_figure(fig):
    buffer = io.BytesIO()
    fig.savefig(buffer, format='png', dpi=110, bbox_inches='tight', pad_inches=.15, facecolor='white')
    plt.close(fig)
    display(Image(data=buffer.getvalue()))
    buffer.close()

LABS = ("LAB019", "LAB020", "LAB021")
FERMENTER = {"LAB019": "F1", "LAB020": "F2", "LAB021": "F3"}
MATCHED_ORIGINAL = {"LAB019": "LAB004", "LAB020": "LAB006", "LAB021": "LAB009"}
COLORS = {"LAB019": "#4c72b0", "LAB020": "#55a868", "LAB021": "#c44e52"}
T0 = pd.Timestamp("2026-09-21 17:00:00")
# Hora y dosis reales del segundo pulso: hoja Nutricion, no un valor heredado del ajuste histórico.
nutrition_wb = openpyxl.load_workbook(
    DATA_DIR / "Mediciones-Offline-LAB019-021.xlsx", read_only=True, data_only=True
)
pulse2_rows = [
    row for row in nutrition_wb["Nutricion"].iter_rows(min_row=2, values_only=True)
    if row[0] in LABS and row[2] == "Pulso 2"
]
nutrition_wb.close()
assert len(pulse2_rows) == len(LABS)
assert {row[0] for row in pulse2_rows} == set(LABS)
PULSE2_DT = {row[0]: pd.Timestamp(row[3]) for row in pulse2_rows}
PULSE2_H = {lab: (stamp - T0).total_seconds() / 3600.0 for lab, stamp in PULSE2_DT.items()}
pulse2_doses = {row[0]: float(row[4]) / 1000.0 for row in pulse2_rows}  # mg/L -> kg/m³
assert len(set(pulse2_doses.values())) == 1, "el modelo usa una dosis común para los tres lotes"
DELTA_N_KG_M3 = next(iter(pulse2_doses.values()))
GRID_H = 0.25
CO2_FACTOR_G_L_H_PER_SCCM = 44.0095 * 60.0 * 0.74 / (1000.0 * 24.16 * 2.0)
assert np.isclose(CO2_FACTOR_G_L_H_PER_SCCM, 0.0404391928807947)
print("ROOT:", ROOT)
print("t0:", T0, "| pulse 2 [h]:", PULSE2_H)

pd.set_option("display.max_rows", 120)

assert Path(sys.prefix).name == 'fermentation', sys.executable
print('Python:', sys.executable)

import hashlib
import time
from scipy.integrate import solve_ivp
from scipy.linalg import svd
OUT = LAB2026 / 'notebooks' / 'EKF' / 'post_activity_prediction_results'
OUT.mkdir(exist_ok=True)
RECENT_LABS = LABS
CONFIG = dict(fd_fraction=.01, fd_check_fraction=.005, primary_cadence_h=3.,
              sigma_floor=.04, sigma_relative_peak=.10, rank_rtol=1e-8, rank_atol=1e-6,
              primary_grid_h=.25, rtol=2e-9, atol=2e-11, max_step_h=.5)
def sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1024*1024), b''): digest.update(block)
    return digest.hexdigest()
PROTECTED = [p for p in DATA_DIR.rglob('*') if p.is_file()] + [
    LAB2026/'results/estimability_historical_natural/theta_natural_full.csv',
    LAB2026/'results/estimability_historical_natural/theta.csv',
    LAB2026/'results/co2_matrix_cross_validation_2026/fit_parameters.csv',
    LAB2026/'notebooks/diagnostics/lab019_021_co2_temperature_shape_audit.ipynb',
    LAB2026/'notebooks/co2_reformulation/02_nutrition_structure_comparison.source.ipynb',
    FM/'shared/run_new_must_glycerol_estimability_doe.py',
    FM/'shared/run_secondary_joint_campaign_doe.py',
    LAB2026/'run_co2_matrix_cross_validation_2026.py',
    FM/'pilot_2025/run_pilot_2025_co2_solubility_integrated_doe.py']
HASH_BEFORE = {str(p.relative_to(ROOT)):sha256(p) for p in PROTECTED}
pd.set_option('display.max_rows',40)

sys.path.insert(0, str(LAB2026/'notebooks/EKF'))
import _post_activity_prediction as diagnostic
PROTECTED += [LAB2026/'notebooks/EKF/co2_local_observability.source.ipynb',
              LAB2026/'notebooks/EKF/_post_activity_prediction.py',
              LAB2026/'nut_sample_audit.py']
HASH_BEFORE = {str(p.relative_to(ROOT)):sha256(p) for p in PROTECTED}
FIGURES = OUT/'figures'
FIGURES.mkdir(exist_ok=True)
print('Diagnóstico retrospectivo; horizonte completo; sin optimizador ni EKF.')

In [ ]:
DEFAULT_PARAMETERS = ('sN','qXG','qXF','sG','sF','m0')
def load_parameters(fm):
    results = fm / 'laboratory_2026' / 'results'
    full_path = results / 'estimability_historical_natural/theta_natural_full.csv'
    old_path = results / 'estimability_historical_natural/theta.csv'
    layer_path = results / 'co2_matrix_cross_validation_2026/fit_parameters.csv'
    full_table = pd.read_csv(full_path)
    full = dict(zip(full_table.parameter, full_table.theta.astype(float)))
    old_raw = pd.read_csv(old_path, index_col=0).iloc[:, 0].astype(float).to_dict()
    defaults = co2_cross._load_theta(old_path)
    assert set(full) == set(base.FULL17)
    assert set(base.FULL17) - set(old_raw) == set(DEFAULT_PARAMETERS)
    defaults = {key: float(defaults[key]) for key in base.FULL17}
    for key in base.FULL17:
        expected = base.DEFAULT_THETA[key] if key in DEFAULT_PARAMETERS else full[key]
        assert np.isclose(defaults[key], expected, rtol=1e-12, atol=0), key
    layer = pd.read_csv(layer_path)
    layer = layer[layer.calibration_matrix.eq('natural')].copy()
    params = dict(zip(layer.parameter, layer.estimate.astype(float)))
    assert set(params) == set(co2_cross.PARAMETER_BOUNDS)
    params.update(A=.30, tau_rise_h=4., tau_decay_h=12.,
                  N_ref_g_l=.14, kernel_normalization='grid_max')
    table = pd.DataFrame({'parameter': base.FULL17,
                          'theta_full': [full[k] for k in base.FULL17],
                          'theta_defaults': [defaults[k] for k in base.FULL17],
                          'origen_defaults': ['DEFAULT_THETA' if k in DEFAULT_PARAMETERS else 'calibración natural histórica' for k in base.FULL17]})
    return {'theta_full': full, 'theta_defaults': defaults}, params, table, layer, (full_path, old_path, layer_path)
THETA_SETS, CO2_PARAMS, THETA_TABLE, CO2_PARAMETER_TABLE, _ = load_parameters(FM)

import ast
nb02 = json.loads((LAB2026/'notebooks/co2_reformulation/02_nutrition_structure_comparison.source.ipynb').read_text(encoding='utf-8'))
param_source = next(''.join(c['source']) for c in nb02['cells']
    if c['cell_type']=='code' and 'PARAM_SETS = {' in ''.join(c['source']))
tree = ast.parse(param_source)
assignment = next(node for node in tree.body if isinstance(node, ast.Assign)
    and any(isinstance(t,ast.Name) and t.id=='PARAM_SETS' for t in node.targets))
old_node = next(v for k,v in zip(assignment.value.keys,assignment.value.values)
                if isinstance(k,ast.Constant) and k.value=='antiguos')
manual = {ast.literal_eval(k):ast.literal_eval(v) for k,v in zip(old_node.keys,old_node.values) if k is not None}
assert manual == dict(A=.30,tau_rise_h=4.,tau_decay_h=12.,kernel_normalization='grid_max')
assert all(CO2_PARAMS[k]==v for k,v in manual.items())
assert 'HISTORICAL_CO2_LAYER_PATH' in param_source
display(THETA_TABLE)
display(pd.DataFrame([{'parameter':k,'value':v} for k,v in CO2_PARAMS.items()]))
print('Parámetros antiguos verificados contra la configuración AST del notebook 02.')

## 2. Mediciones y reloj físico
Se conservan la hora de extracción y el origen de cada muestra. m1 fue tomada 1,17 h antes de la inoculación y se mantiene el mapeo histórico a t=0 solamente para reproducir la referencia. El etanol inicial medido no se sustituye por cero. Las muestras NUT se muestran con su intervalo provisional; no inicializan ni puntúan un residuo a una hora exacta inventada.

La temperatura medida futura se utiliza como entrada conocida retrospectivamente. El suavizado centrado de la vista de temperatura y las medianas horarias del logger no constituyen un detector online validado. Para inicializar Cd sólo se usan bins horarios finalizados: etiqueta t implica disponibilidad a t+1 h. Las máscaras y el cero del sensor son los de la auditoría histórica.

In [ ]:
def read_csv_window(path, usecols, start, end, chunksize=500_000):
    frames = []
    for chunk in pd.read_csv(path, usecols=usecols, chunksize=chunksize, low_memory=False):
        stamp = pd.to_datetime(chunk["timestamp"], errors="coerce")
        keep = stamp.between(start, end)
        if keep.any():
            part = chunk.loc[keep].copy()
            part["timestamp"] = stamp.loc[keep]
            frames.append(part)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=usecols)


window_start, window_end = T0 - pd.Timedelta(hours=6), T0 + pd.Timedelta(hours=170)
temperature_raw = read_csv_window(
    DATA_DIR / "backup_global_temperatura.csv", ["timestamp", "fermentador", "T", "SP"], window_start, window_end
)
for col in ("T", "SP"):
    temperature_raw[col] = pd.to_numeric(temperature_raw[col], errors="coerce")
co2_raw = read_csv_window(
    DATA_DIR / "backup_global_co2.csv", ["timestamp", "fermentador", "flow_sccm", "flow_filt_sccm", "status"], window_start, window_end
)
for col in ("flow_sccm", "flow_filt_sccm"):
    co2_raw[col] = pd.to_numeric(co2_raw[col], errors="coerce")

TEMP_RESAMPLE = "1min"    # mediana por minuto: elimina outliers y uniformiza el cadence
TEMP_MEDIAN = "20min"     # mediana móvil: suprime ráfagas y spikes del control on/off
TEMP_MEAN = "45min"       # media móvil: suaviza la oscilación de banda del control

temp_log = {}
for _lab in LABS:
    _f = FERMENTER[_lab]
    frame = (temperature_raw[temperature_raw["fermentador"].eq(_f)]
             .dropna(subset=["T"]).sort_values("timestamp")
             .set_index("timestamp")[["T", "SP"]])
    frame = frame.resample(TEMP_RESAMPLE).median()
    t_med = frame["T"].rolling(TEMP_MEDIAN, center=True, min_periods=1).median()
    temp_log[_f] = pd.DataFrame({
        "T_filt": t_med.rolling(TEMP_MEAN, center=True, min_periods=1).mean(),
        "SP": frame["SP"],
    })

In [ ]:
from shared.new_must_data_loader import ETHANOL_G_L_PER_PERCENT_VV


def numeric(value):
    try:
        out = float(value)
        return out if np.isfinite(out) else np.nan
    except (TypeError, ValueError):
        return np.nan


def load_offline_samples():
    path = DATA_DIR / "Mediciones-Offline-LAB019-021.xlsx"
    wb = openpyxl.load_workbook(path, read_only=True, data_only=True)
    rows = []
    for excel_row, row in enumerate(wb["MedicionesOffline"].iter_rows(min_row=2, values_only=True), start=2):
        fecha, hora, lab, brix, density = row[0], row[1], row[2], row[5], row[6]
        if lab is None and brix is None and density is None:
            continue
        if lab not in LABS or fecha is None or hora is None or row[4] is None:
            raise ValueError(f"MedicionesOffline fila {excel_row}: metadatos de muestra incompletos")
        hh = hora.time() if isinstance(hora, datetime) else hora
        rows.append({
            "lab": str(lab), "sample": str(row[4]).strip(),
            "sample_timestamp": datetime.combine(fecha.date() if hasattr(fecha, "date") else fecha, hh),
            "Brix": numeric(brix), "density": numeric(density),
            "Etanol_pct_vv": numeric(row[10]), "metadata_fallback": False,
        })
    wb.close()
    result = pd.DataFrame(rows)
    assert not result.duplicated(["lab", "sample"]).any(), "muestras offline duplicadas"
    return result

def load_y15():
    rows = []
    paths = sorted(DATA_DIR.glob("EXPAuto*.txt")) + sorted(DATA_DIR.glob("ONLINE*.txt"))
    for path in paths:
        for line in open(path, encoding="utf-8-sig", errors="replace"):
            fields = line.rstrip("\n").split("\t")
            if len(fields) < 6:
                continue
            raw = fields[3].strip()
            value = float(raw.replace(",", ".")) if re.fullmatch(r"-?\d+(,\d+)?", raw) else np.nan
            rows.append({"sample_id": fields[0], "analyte": fields[1], "value": value, "raw_value": raw, "analysis_timestamp": fields[5], "source_file": path.name})
    long = pd.DataFrame(rows).drop_duplicates(["sample_id", "analyte"], keep="first")
    wide = long.pivot(index="sample_id", columns="analyte", values="value")
    raw_wide = long.pivot(index="sample_id", columns="analyte", values="raw_value")
    return long, wide, raw_wide


offline = load_offline_samples()
# Dilución Alcolyzer m12, distinta por lote (cuaderno de laboratorio): LAB020 27.5 mL de vino + 10 mL de agua;
# LAB021 32.5 mL de vino + 5 mL de agua (ambas a 37.5 mL). Sólo corrige etanol (medido en la mezcla);
# Brix y densidad se midieron antes de diluir.
DILUTION_FACTORS_M12 = {"LAB020": 37.5 / 27.5, "LAB021": 37.5 / 32.5}
for _lab_m12, _factor in DILUTION_FACTORS_M12.items():
    _sel = offline["lab"].eq(_lab_m12) & offline["sample"].eq("12")
    offline.loc[_sel, "Etanol_pct_vv"] = offline.loc[_sel, "Etanol_pct_vv"] * _factor
assert np.isclose(offline.loc[(offline["lab"] == "LAB020") & (offline["sample"] == "12"), "Etanol_pct_vv"].iloc[0], 7.33 * 15 / 11)
assert np.isclose(offline.loc[(offline["lab"] == "LAB021") & (offline["sample"] == "12"), "Etanol_pct_vv"].iloc[0], 8.66 * 15 / 13)
y15_long, y15, y15_raw = load_y15()

chem_rows = []
for lab in LABS:
    samples = offline[offline["lab"].eq(lab)].copy().set_index("sample")
    numbered_samples = sorted((item for item in samples.index if str(item).isdigit()), key=int)
    assert "1" in numbered_samples, f"{lab}: falta muestra inicial"
    for sample in numbered_samples:
        sid = f"{lab}-{sample}"
        meta = samples.loc[sample]
        stamp = pd.Timestamp(meta["sample_timestamp"])
        sample_t_h = (stamp - T0).total_seconds() / 3600.0
        t_h = 0.0 if sample == "1" else sample_t_h
        get = lambda name: float(y15.loc[sid, name]) if sid in y15.index and name in y15.columns and pd.notna(y15.loc[sid, name]) else np.nan
        get_raw = lambda name: str(y15_raw.loc[sid, name]) if sid in y15_raw.index and name in y15_raw.columns and pd.notna(y15_raw.loc[sid, name]) else ""
        gf_320_raw = get("GLU-FRU-320")
        gf_low_raw = get("GLUCOSE-FRUCTOSE")
        if np.isfinite(gf_320_raw) and gf_320_raw >= 0.0:
            gf, gf_source = gf_320_raw, "GLU-FRU-320"
        elif np.isfinite(gf_low_raw) and gf_low_raw >= 0.0:
            gf, gf_source = gf_low_raw, "GLUCOSE-FRUCTOSE"
        else:
            gf, gf_source = np.nan, ""
        glucose_320_raw = get("GLUCOSE-320")
        glucose_low_raw = get("GLUCOSE")
        glucose_raw = glucose_320_raw if np.isfinite(glucose_320_raw) else glucose_low_raw
        glucose = glucose_raw if np.isfinite(glucose_raw) and glucose_raw >= 0.0 else np.nan
        fructose = gf - glucose if np.isfinite(gf) and np.isfinite(glucose) and gf >= glucose else np.nan
        pan, ammonia_raw = get("PAN"), get("AMMONIA")
        ammonia = max(ammonia_raw, 0.0) if np.isfinite(ammonia_raw) else np.nan
        yan = pan + 0.823 * ammonia if np.isfinite(pan) and np.isfinite(ammonia) else np.nan
        chem_rows.append({
            "lab": lab, "sample": sample, "sample_timestamp": stamp, "sample_t_h": sample_t_h, "t_h": t_h,
            "metadata_fallback": bool(meta["metadata_fallback"]), "density": float(meta["density"]), "Brix": float(meta["Brix"]),
            "GF_g_l": gf, "GF_source": gf_source,
            "GF_320_raw_g_l": gf_320_raw, "GF_low_raw_g_l": gf_low_raw,
            "GF_320_below_zero": bool(np.isfinite(gf_320_raw) and gf_320_raw < 0.0),
            "G_g_l": glucose, "G_raw_g_l": glucose_raw, "F_g_l": fructose,
            "PAN_mg_l": pan, "NH3_raw_mg_l": ammonia_raw, "NH3_censored_mg_l": ammonia,
            "YAN_mg_l": yan, "N_kg_m3": yan / 1000.0 if np.isfinite(yan) else np.nan,
            "Gly_g_l": get("GLYCEROL"), "Pyr_mg_l": get("PYRUVIC ACID"),
            "Etanol_pct_vv": float(meta["Etanol_pct_vv"]),
            "E_g_l": meta["Etanol_pct_vv"] * ETHANOL_G_L_PER_PERCENT_VV if np.isfinite(meta["Etanol_pct_vv"]) else np.nan,
            "AcAld_mg_l": get("ACETALDEHID-1000"), "Acetate_g_l": get("ACETIC ACID"),
            "G_raw_text": get_raw("GLUCOSE-320") or get_raw("GLUCOSE"),
            "GF_raw_text": get_raw("GLU-FRU-320"),
            "GF_low_raw_text": get_raw("GLUCOSE-FRUCTOSE"),
            "NH3_below_zero": bool(np.isfinite(ammonia_raw) and ammonia_raw < 0.0),
        })
from laboratory_2026.nut_sample_audit import load_nut_samples
NUT_SAMPLE_TIMES = {}  # completar sólo con horas de extracción confirmadas
NUT_SAMPLES = load_nut_samples(DATA_DIR, {lab:T0 for lab in LABS}, NUT_SAMPLE_TIMES)
NUT_SAMPLES.to_csv(OUT/'nut_sample_audit.csv', index=False)
for nut in NUT_SAMPLES.itertuples():
    chem_rows.append(dict(lab=nut.lab, sample='NUT', sample_timestamp=nut.sample_timestamp,
        sample_t_h=nut.plot_t_h, t_h=nut.plot_t_h, metadata_fallback=nut.time_provisional,
        PAN_mg_l=nut.PAN_mg_l, NH3_raw_mg_l=nut.NH3_raw_mg_l,
        NH3_censored_mg_l=nut.NH3_censored_mg_l, YAN_mg_l=nut.YAN_mg_l, N_kg_m3=nut.N_kg_m3,
        YAN_reported_mg_l=nut.YAN_reported_mg_l, NH3_below_zero=False,
        time_lower_h=nut.time_lower_h, time_upper_h=nut.time_upper_h,
        time_provisional=nut.time_provisional, time_source=nut.time_source))
chem = pd.DataFrame(chem_rows).sort_values(["lab", "t_h"]).reset_index(drop=True)
chem['is_NUT'] = chem['sample'].eq('NUT')
chem['time_provisional'] = chem['time_provisional'].fillna(False).astype(bool)
chem['time_lower_h'] = chem['time_lower_h'].fillna(chem.t_h)
chem['time_upper_h'] = chem['time_upper_h'].fillna(chem.t_h)
chem['time_source'] = chem['time_source'].fillna('offline extraction timestamp; m1 mapped to t0')
chem["is_initial"] = chem["sample"].eq("1")
chem["soft_flag_GF"] = chem["lab"].eq("LAB019") & chem["sample"].eq("4")
chem["soft_flag_metadata"] = chem["metadata_fallback"]
chem["y15_available"] = chem[["GF_g_l", "YAN_mg_l", "Gly_g_l", "Pyr_mg_l", "AcAld_mg_l"]].notna().any(axis=1)
assert chem[chem["sample"].eq("1")]["E_g_l"].notna().all(), "falta Etanol m1 (inicializa E0)"

assert np.allclose(chem[chem["sample"].eq("1")]["t_h"], 0.0)
assert chem.query("lab == 'LAB020' and sample == '4'")["G_g_l"].isna().all()
assert chem.query("lab == 'LAB020' and sample == '4'")["G_raw_g_l"].iloc[0] == -7.28
assert chem[chem["sample"].eq("12")]["y15_available"].all(), "falta exportación Y15 m12"
assert chem.loc[chem["sample"].eq("12"), "GF_320_below_zero"].all(), "revisar QA de GF m12"
display(chem[["lab", "sample", "sample_t_h", "t_h", "GF_g_l", "G_g_l", "F_g_l", "YAN_mg_l", "Gly_g_l", "E_g_l", "Pyr_mg_l", "AcAld_mg_l", "y15_available", "NH3_below_zero", "metadata_fallback"]])
print("Muestras sin resultado Y15:", chem.loc[~chem["y15_available"], ["lab", "sample"]].to_dict("records"))

# Todas las mediciones Oculyze: concentración total en millones de células/mL,
# conversión de 30 pg/célula y viabilidad porcentual para comparar con X viable.
from shared.new_must_data_loader import MILLION_CELLS_ML_TO_KG_M3
biomass_rows=[]
for lab in LABS:
    report_path=DATA_DIR/f'Oculyze_report_{lab}.csv'
    with open(report_path,encoding='utf-8-sig') as handle:
        report_rows=list(csv.DictReader(handle))
    for row in report_rows:
        sample_id=row['Name'].strip().upper()
        if sample_id=='CERO':
            continue  # Se conserva la convención primaria de concentración cruda.
        match=re.fullmatch(rf'{lab}-(\d+)',sample_id)
        if match is None:
            raise ValueError(f'{report_path.name}: identificador inesperado {sample_id}')
        sample=str(int(match.group(1)))
        meta=chem[chem.lab.eq(lab)&chem['sample'].eq(sample)]
        assert len(meta)==1,(lab,sample)
        meta=meta.iloc[0]
        concentration=float(row['Concentration'])
        viability=float(row['Viability'])
        assert np.isfinite(concentration) and concentration>=0
        assert np.isfinite(viability) and 0<=viability<=100
        total=concentration*MILLION_CELLS_ML_TO_KG_M3
        viable=total*viability/100.
        biomass_rows.append(dict(lab=lab,sample=sample,sample_id=sample_id,
            sample_timestamp=meta.sample_timestamp,sample_t_h=float(meta.sample_t_h),
            t_h=float(meta.t_h),concentration_mcells_ml=concentration,viability_pct=viability,
            X_total_g_l=total,X_viable_g_l=viable,Xd_g_l=total-viable,
            source_file=report_path.name))
BIOMASS_OBSERVATIONS=pd.DataFrame(biomass_rows).sort_values(['lab','t_h']).reset_index(drop=True)
assert not BIOMASS_OBSERVATIONS.duplicated(['lab','sample']).any()
assert BIOMASS_OBSERVATIONS.groupby('lab').size().to_dict()=={lab:11 for lab in LABS}
assert np.allclose(BIOMASS_OBSERVATIONS.X_total_g_l,
                   BIOMASS_OBSERVATIONS.X_viable_g_l+BIOMASS_OBSERVATIONS.Xd_g_l)
chem=chem.merge(BIOMASS_OBSERVATIONS[['lab','sample','X_total_g_l','X_viable_g_l','Xd_g_l']],
                on=['lab','sample'],how='left',validate='one_to_one')
display(BIOMASS_OBSERVATIONS[['lab','sample','sample_t_h','t_h','concentration_mcells_ml',
                             'viability_pct','X_total_g_l','X_viable_g_l','Xd_g_l']].round(5))
print('Mediciones Oculyze por lote:',BIOMASS_OBSERVATIONS.groupby('lab').size().to_dict())

assert chem[chem.is_NUT].lab.nunique() == 3
assert chem.loc[chem.is_NUT, ['GF_g_l','G_g_l','F_g_l','Gly_g_l','E_g_l','X_viable_g_l']].isna().all().all()
chem.to_csv(OUT/'chemistry_samples_with_NUT.csv', index=False)
display(NUT_SAMPLES[['lab','YAN_reported_mg_l','YAN_mg_l','pulse_timestamp',
                     'analysis_PAN','analysis_YAN','sample_timestamp','time_lower_h','plot_t_h','time_upper_h']])
def draw_nut_state(ax, lab, state):
    if state != 'N': return
    nut = NUT_SAMPLES[NUT_SAMPLES.lab.eq(lab)].iloc[0]
    x = float(nut.plot_t_h)
    ax.errorbar([x],[nut.N_kg_m3],
        xerr=[[x-nut.time_lower_h],[nut.time_upper_h-x]],fmt='*',color='#8e24aa',
        ms=8,capsize=3,zorder=8,label='NUT · tiempo provisional' if nut.time_provisional else 'NUT · hora confirmada')

In [ ]:
co2_hourly_parts, mask_rows = [], []
for lab in LABS:
    f = FERMENTER[lab]
    frame = co2_raw[co2_raw["fermentador"].eq(f)].sort_values("timestamp").copy()
    frame["t_h"] = (frame["timestamp"] - T0).dt.total_seconds() / 3600.0
    baseline = frame[frame["t_h"].between(-2.0, -0.5)]["flow_sccm"].dropna().to_numpy(dtype=float)
    baseline_center = float(np.median(baseline))
    sigma = float(1.4826 * np.median(np.abs(baseline - baseline_center)))
    threshold = max(3.5 * sigma, 0.5)
    search = frame[frame["t_h"].between(-0.5, 0.75)].copy()
    crossing = search[np.abs(search["flow_sccm"] - baseline_center) > threshold]
    if crossing.empty:
        event_start_h, event_raw_end_h = 0.0, 1.0
    else:
        event_start_h, event_raw_end_h = float(crossing["t_h"].min()), float(crossing["t_h"].max())
    mask_end_h = max(0.0, event_raw_end_h) + 1.5

    zero_support = frame[frame["t_h"].between(mask_end_h, 12.0)]["flow_filt_sccm"].dropna()
    zero_sccm = max(0.0, float(zero_support.quantile(0.10))) if len(zero_support) else 0.0
    frame["flow_corrected_sccm"] = np.maximum(frame["flow_filt_sccm"] - zero_sccm, 0.0)
    hourly = frame.set_index("timestamp")[["flow_corrected_sccm", "flow_filt_sccm", "flow_sccm"]].resample("1h").median().dropna(subset=["flow_corrected_sccm"]).reset_index()
    hourly["t_h"] = (hourly["timestamp"] - T0).dt.total_seconds() / 3600.0
    hourly = hourly[hourly["t_h"].between(0.0, 170.0)].copy()
    hourly["lab"] = lab
    hourly["co2_observed_g_l_h"] = hourly["flow_corrected_sccm"] * CO2_FACTOR_G_L_H_PER_SCCM
    hourly["co2_valid"] = hourly["t_h"] > mask_end_h
    hourly["left_censored"] = hourly["co2_observed_g_l_h"] < co2_cross.CO2_DETECTION_LIMIT_G_L_H
    co2_hourly_parts.append(hourly)
    mask_rows.append({
        "lab": lab, "fermenter": f, "baseline_raw_sccm": baseline_center, "baseline_sigma_sccm": sigma,
        "artifact_threshold_sccm": threshold, "event_start_h": event_start_h,
        "event_raw_end_h": event_raw_end_h, "recovery_tail_min": 90, "mask_end_h": mask_end_h,
        "sensor_zero_sccm": zero_sccm,
    })

co2_hourly = pd.concat(co2_hourly_parts, ignore_index=True)
mask_audit = pd.DataFrame(mask_rows)
display(mask_audit.round(4))
from shared.new_must_data_loader import MILLION_CELLS_ML_TO_KG_M3
def oculyze_m1_biomass(lab):
    with open(DATA_DIR / f"Oculyze_report_{lab}.csv", encoding="utf-8-sig") as handle:
        rows = list(csv.DictReader(handle))
    m1 = next(r for r in rows if r["Name"].strip().upper() == f"{lab}-1")
    cero = next((r for r in rows if r["Name"].strip().upper() == "CERO"), None)
    concentration = float(m1["Concentration"])
    viability = float(m1["Viability"])
    return {
        "x0_kg_m3": concentration * MILLION_CELLS_ML_TO_KG_M3,
        "xd0_kg_m3": concentration * MILLION_CELLS_ML_TO_KG_M3 * (1.0 - viability / 100.0),
        "oculyze_m1_concentration_mcells_ml": concentration,
        "oculyze_m1_viability_pct": viability,
        "x0_cero_subtracted_kg_m3": (
            max(concentration - float(cero["Concentration"]), 0.0) * MILLION_CELLS_ML_TO_KG_M3
            if cero is not None else float("nan")
        ),
    }

recent_batches = {}
recent_time_grids = {}
recent_initial_rows = []
for lab in RECENT_LABS:
    measured = chem[chem.lab.eq(lab)].sort_values('t_h')
    init = measured[measured['sample'].eq('1')].iloc[0]
    biomass = oculyze_m1_biomass(lab)
    support = co2_hourly[co2_hourly.lab.eq(lab)].sort_values('t_h')
    horizon = float(support.t_h.max())
    batch_horizon = max(horizon, float(measured.t_h.max()))
    grid = np.arange(0., batch_horizon + GRID_H / 2., GRID_H)
    grid = np.unique(np.r_[grid, measured.loc[~measured.is_NUT, "t_h"].to_numpy(float), PULSE2_H[lab]])
    grid = grid[(grid >= 0.) & (grid <= batch_horizon + 1e-9)]
    temp = temperature_raw[temperature_raw.fermentador.eq(FERMENTER[lab])].dropna(subset=['T']).sort_values('timestamp')
    temp_h = (temp.timestamp - T0).dt.total_seconds().to_numpy() / 3600.
    temperature = np.interp(grid, temp_h, temp['T'].to_numpy(float))
    initials = {'X': biomass['x0_kg_m3'], 'Xd': biomass['xd0_kg_m3'],
        'N': float(init.N_kg_m3), 'G': float(init.G_g_l), 'F': float(init.F_g_l),
        'E': float(init.E_g_l), 'Gly': float(init.Gly_g_l)}
    assert np.isfinite(list(initials.values())).all()
    recent_batches[lab] = base.BatchData(medium='natural',batch=lab,time=grid,
        temperature_c=temperature, initials=initials,
        pulses={**{channel: tuple() for channel in base.INPUT_CHANNELS}, 'N': ((PULSE2_H[lab],DELTA_N_KG_M3),)},
        observations={state: np.full(len(grid),np.nan) for state in base.STATE_NAMES})
    cache_time = np.arange(0.,horizon+co2_cross.DRIVER_GRID_H+1e-9,co2_cross.DRIVER_GRID_H)
    cache_time = cache_time[cache_time <= horizon+1e-9]
    if not np.isclose(cache_time[-1],horizon): cache_time = np.append(cache_time,horizon)
    recent_time_grids[lab] = cache_time
    recent_initial_rows.append({'lab':lab, **initials, 'X_source':'Oculyze m1 crudo × 0.03',
        'chemistry_source':'Y15/Alcolyzer m1', 'temperature_extrapolated_h':max(0.,batch_horizon-float(temp_h.max()))})
display(pd.DataFrame(recent_initial_rows))

## 3. Selección de los dos inicios y supuestos de inicialización
Se eligen muestras existentes por su disponibilidad química, no buscando el menor error futuro. Se comprueba que los tres bins horarios completos previos tienen CO₂ ≥0,05 g/L/h. Esto confirma actividad detectable antes de la muestra, pero no identifica el fin biológico del lag.

- **m7, 46 h:** N, GF, E, Gly medidos; reparto G/F heredado de la última muestra anterior válida y no marcada. Los resultados de G y F son condicionales a ese reparto.
- **m10, 70,5 h:** N, G, F, E, Gly disponibles en la misma muestra; F se obtiene como GF−G. Esta muestra contiene más información inicial y un horizonte posterior más corto: sus errores no se comparan directamente con los de m7 para elegir un ganador.
- **X y Xd:** proxy Oculyze total y fracción no viable, siguiendo la conversión histórica; actividad no equivale a viabilidad. X se perturba ×0,5 y ×2 y también se sustituye por el X propagado nominal.
- **Cd:** inversión del flujo anterior con parámetros congelados; no medición de CO₂ disuelto. Se prueban Cd=0, ×0,5 y ×2.
- **O₂:** valor nominal propagado, no medido; se contrasta con saturación al reinicio.

In [ ]:
SAMPLES = ('7','10')
STARTS = {}; INIT_ROWS=[]
for lab in LABS:
    for sample in SAMPLES:
        values, audit = diagnostic.initial_from_sample(chem,lab,sample)
        STARTS[(lab,sample)] = values
        audit.update(pulse_h=PULSE2_H[lab], pulse_tau_h=PULSE2_H[lab]-audit['start_h'],
                     pulse_already_occurred=PULSE2_H[lab]<audit['start_h'])
        INIT_ROWS.append(audit)
INITIALIZATION = pd.DataFrame(INIT_ROWS)
display(INITIALIZATION[['lab','sample','start_h','pulse_h','pulse_tau_h',
                       'GF_direct','GF_source','fraction_source_time_h','X','N','G','F','E']])
print('m7 a las 46 h es anterior a los tres pulsos; m10 conserva historia previa en LAB019 y LAB021.')

## 4. Continuación del mismo modelo y memoria de nutrición
El cambio químico inicial se realiza una sola vez. Después se integra la RHS oficial, con la temperatura en su hora real y con saltos exactos de nutrición. El gate nominal se conserva en su reloj original: **no se vuelve a introducir un lag al reiniciar**. El control de continuación compara el gas después del primer punto: el esquema original guarda en ese punto el flujo del paso previo, mientras que el reinicio muestra el flujo instantáneo del Cd inicial. Esa diferencia de convención se registra por separado y no afecta los pasos posteriores.

La capa histórica utiliza dos trayectorias (con/sin nutrición), una rampa y un multiplicador. Después de un pulso no basta con reiniciar siete estados. Para m10 conservamos el contraste contrafactual nominal en tₐ:

\[
x_{\mathrm{sinN}}^{nuevo}(t_a)
=x_{\mathrm{conN}}^{nuevo}(t_a)
-[x_{\mathrm{conN}}^{nom}(t_a)-x_{\mathrm{sinN}}^{nom}(t_a)].
\]

Ese contraste es información **del modelo**, no una medición. Si exige estados negativos se informa el recorte; la sensibilidad a borrar el contraste se calcula por separado. La rampa, el transitorio y la normalización antigua `grid_max` se heredan del horizonte original. Esta última sigue siendo una convención retrospectiva y no una formulación online final.

El CO₂ disuelto y O₂ conservan su dinámica; el gas se inicializa con flujo pasado. No se reajusta ganancia, liberación ni cinética con mediciones futuras.

In [ ]:
BASELINES={}; CONTROLS=[]; ALL_CURVES={}; METRIC_ROWS=[]; CHECK_ROWS=[]; GAS_INIT=[]
for theta_name,th in THETA_SETS.items():
    for lab in LABS:
        batch=recent_batches[lab]
        end=float(batch.time[-1])
        time_grid=np.unique(np.r_[np.arange(0.,end+.125,.25),batch.time])
        time_grid=time_grid[time_grid<=end+1e-9]
        ref=diagnostic.baseline(batch,th,time_grid,CO2_PARAMS)
        BASELINES[(lab,theta_name)]=ref
        official=base.simulate(batch,th,time_grid)
        assert official is not None
        core_error=float(np.max(np.abs(official.to_numpy()-ref['core'].to_numpy())))
        assert core_error<.005,core_error
        rate_error=float(np.max(np.abs(diagnostic.production(th,batch,ref['core'])-
            co2_cross.co2_model.co2_production_g_l_h(th,batch,ref['core'],time_grid))))
        assert rate_error<1e-12
        observed=co2_hourly[co2_hourly.lab.eq(lab)]
        for sample in SAMPLES:
            audit=INITIALIZATION.query('lab==@lab and sample==@sample').iloc[0]
            start=float(audit.start_h)
            nominal=ref['frame'].loc[start]
            unchanged=diagnostic.restart(batch,th,ref,start,
                {s:float(nominal[s]) for s in base.STATE_NAMES},nominal.Cd,nominal.O2,CO2_PARAMS)
            continuation_error=float(np.max(np.abs(unchanged[list(base.STATE_NAMES)+['predicted','Cd','O2']].iloc[1:].to_numpy()-
                ref['frame'].loc[unchanged.index,list(base.STATE_NAMES)+['predicted','Cd','O2']].iloc[1:].to_numpy())))
            chemistry_continuation_error=float(np.max(np.abs(unchanged[list(base.STATE_NAMES)].to_numpy()-
                ref['frame'].loc[unchanged.index,list(base.STATE_NAMES)].to_numpy())))
            gas_continuation_error=float(np.max(np.abs(unchanged.predicted.iloc[1:]-ref['frame'].loc[unchanged.index].predicted.iloc[1:])))
            initial_gas_convention_error=float(abs(unchanged.predicted.iloc[0]-nominal.predicted))
            assert chemistry_continuation_error<5e-4,chemistry_continuation_error
            assert gas_continuation_error<1e-5,gas_continuation_error
            # Pure clock relabelling must reproduce the exact same chemistry.
            local_batch=replace(batch,time=batch.time-start,
                pulses={k:tuple((t-start,d) for t,d in v if t>=start) for k,v in batch.pulses.items()},
                initials={s:float(nominal[s]) for s in base.STATE_NAMES})
            local=diagnostic.integrate_core(local_batch,th,unchanged.index.to_numpy()-start)
            clock_error=float(np.max(np.abs(local.to_numpy()-unchanged[list(base.STATE_NAMES)].to_numpy())))
            assert clock_error<5e-4,clock_error
            CONTROLS.append(dict(lab=lab,theta=theta_name,sample=sample,start_h=start,
                unchanged_restart_max_error=continuation_error,clock_relabelling_max_error=clock_error,
                chemistry_continuation_error_g_l=chemistry_continuation_error,
                gas_continuation_error_g_l_h=gas_continuation_error,
                initial_gas_convention_error_g_l_h=initial_gas_convention_error,
                official_core_max_error=core_error,official_production_max_error=rate_error))
            initials=STARTS[(lab,sample)]
            init_core=pd.DataFrame([initials],index=[start])
            saturation=float(diagnostic.csat(batch,init_core,CO2_PARAMS)[0])
            cd,flow_audit=diagnostic.flow_initialization(observed,start,saturation,CO2_PARAMS)
            o2sat=co2_cross.co2_model.o2_saturation_mg_l(base.temperature_at(batch,start),
                initials['E'],initials['G'],initials['F'],1.)
            GAS_INIT.append(dict(lab=lab,theta=theta_name,sample=sample,start_h=start,
                Cd_initial_g_l=cd,O2_initial_mg_l=float(nominal.O2),**flow_audit))
            candidates=diagnostic.scenarios(initials,cd,float(nominal.O2),o2sat,
                audit.GF_direct,float(nominal.X))
            for name,scenario in candidates.items():
                curve=diagnostic.restart(batch,th,ref,start,scenario['initials'],
                    scenario['cd'],scenario['oxygen'],CO2_PARAMS)
                key=(lab,theta_name,sample,name)
                ALL_CURVES[key]=curve
                CHECK_ROWS.append(dict(lab=lab,theta=theta_name,sample=sample,scenario=name,
                    counterfactual_clipped=json.dumps(curve.attrs['counterfactual_clipped']),
                    **diagnostic.checks(curve,batch,th)))
                for row in diagnostic.scores(curve,chem[chem.lab.eq(lab)],observed,start,PULSE2_H[lab]):
                    METRIC_ROWS.append(dict(lab=lab,theta=theta_name,sample=sample,
                        start_h=start,scenario=name,**row))
            # Same future observations for the reference and each restarted scenario.
            for row in diagnostic.scores(ref['frame'],chem[chem.lab.eq(lab)],observed,start,PULSE2_H[lab]):
                METRIC_ROWS.append(dict(lab=lab,theta=theta_name,sample=sample,
                    start_h=start,scenario='referencia_desde_m1',**row))
            print(lab,theta_name,'m'+sample,':',len(candidates),'escenarios; final',end,'h')
METRICS=pd.DataFrame(METRIC_ROWS); CHECKS=pd.DataFrame(CHECK_ROWS)
CONTROL_TABLE=pd.DataFrame(CONTROLS); GAS_INITIALIZATION=pd.DataFrame(GAS_INIT)
assert GAS_INITIALIZATION.activity_confirmed.all()
assert (GAS_INITIALIZATION.gas_init_last_bin_end_h<=GAS_INITIALIZATION.start_h).all()
display(GAS_INITIALIZATION.round(5))
display(CONTROL_TABLE)
display(CHECKS.groupby(['theta','sample'])[['mass_balance_max_g_l','nitrogen_inventory_max_g_l','final_h']].max())

## 5. Actividad observada e inicio elegido
Los círculos muestran caudal observado válido; las franjas indican los intervalos de gas usados para inicializar Cd. Las líneas verticales señalan las muestras ya existentes de 46 y 70,5 h y el pulso registrado. El eje mantiene el reloj de inoculación. No se necesita ninguna nueva toma de muestra sincronizada con un detector.

In [ ]:
def save_figure(fig,name):
    fig.savefig(FIGURES/(name+'.png'),dpi=130,bbox_inches='tight',facecolor='white')
    display_inline_figure(fig)
THETA_COLORS={'theta_full':'#2166ac','theta_defaults':'#d6604d'}
fig,axes=plt.subplots(3,1,figsize=(13,8),layout='constrained')
for ax,lab in zip(axes,LABS):
    gas=co2_hourly.query('lab==@lab and co2_valid')
    ax.plot(gas.t_h,gas.co2_observed_g_l_h,'.-',color='black',ms=3,label='CO₂ observado')
    ax.axhline(.05,color='gray',ls=':',label='Umbral de comprobación')
    for sample,color in [('7','#00897b'),('10','#7b1fa2')]:
        start=float(INITIALIZATION.query('lab==@lab and sample==@sample').start_h.iloc[0])
        ax.axvline(start,color=color,label='Inicio m'+sample)
        ax.axvspan(start-3,start,color=color,alpha=.12)
    ax.axvline(PULSE2_H[lab],color='#d81b60',ls='--',label='Pulso N registrado')
    ax.set(xlim=(0,90),ylabel='g CO₂/L/h',title=lab)
axes[-1].set_xlabel('Horas desde inoculación')
axes[0].legend(ncol=4,fontsize=8)
save_figure(fig,'01_activity_and_existing_samples')

## 6. Predicción completa del CO₂ desde cada inicialización
Gris discontinuo: referencia desde m1. Azul y rojo: reinicio con química de la muestra seleccionada, para los dos theta. Las líneas de reinicio comienzan en la muestra y continúan hasta el último tiempo del experimento, incluyendo el pulso si todavía no ocurrió. Que una curva coincida al inicio con el flujo no es validación: Cd utilizó gas anterior para inicializarse.

In [ ]:
fig,axes=plt.subplots(3,2,figsize=(14,10),layout='constrained')
for i,lab in enumerate(LABS):
    obs=co2_hourly.query('lab==@lab and co2_valid')
    for j,sample in enumerate(SAMPLES):
        ax=axes[i,j]; start=INITIALIZATION.query('lab==@lab and sample==@sample').start_h.iloc[0]
        ax.plot(obs.t_h,obs.co2_observed_g_l_h,'k.',ms=3,label='CO₂ observado')
        for theta_name in THETA_SETS:
            ref=BASELINES[(lab,theta_name)]['frame']
            ax.plot(ref.index,ref.predicted,color=THETA_COLORS[theta_name],ls='--',alpha=.35,
                    label=theta_name+' · desde m1')
            cur=ALL_CURVES[(lab,theta_name,sample,'central')]
            ax.plot(cur.index,cur.predicted,color=THETA_COLORS[theta_name],lw=1.8,
                    label=theta_name+' · reinicio')
        ax.axvline(start,color='#00897b',ls=':',label='Inicio de predicción')
        ax.axvline(PULSE2_H[lab],color='#d81b60',ls=':',label='Pulso N')
        ax.set(title=f'{lab} · m{sample}, tₐ={start:g} h',xlim=(0,170),ylabel='g CO₂/L/h',xlabel='t desde inoculación [h]')
axes[0,0].legend(fontsize=7,ncol=2)
save_figure(fig,'02_co2_full_horizon')

## 7. Química posterior: inicialización y contraste reservado
Rombo verde: cantidad medida utilizada para inicializar. Triángulo naranja: reparto G/F supuesto en m7, **no medición**. Puntos negros: muestras posteriores reservadas. Gris: muestras anteriores al inicio. Las NUT se conservan como estrellas con intervalo temporal provisional.

No se puntúa la muestra de inicialización. No se inventan G/F al final cuando los exports están fuera de rango. N se muestra en mg N/L; el estado interno y el CSV de métricas usan g N/L. La biomasa no se usa para decidir si el modelo fue validado.

In [ ]:
PLOT_CHANNELS=('N','GF','G','F','E')
for sample in SAMPLES:
    fig,axes=plt.subplots(3,5,figsize=(19,10),layout='constrained')
    for i,lab in enumerate(LABS):
        subset=chem[chem.lab.eq(lab)]
        audit=INITIALIZATION.query('lab==@lab and sample==@sample').iloc[0]
        start=float(audit.start_h)
        for j,ch in enumerate(PLOT_CHANNELS):
            ax=axes[i,j]; scale=1000. if ch=='N' else 1.
            col=diagnostic.CHANNELS[ch]
            data=subset.loc[~subset.is_NUT & subset[col].notna()]
            before=data[data.t_h<start]; after=data[data.t_h>start]
            ax.scatter(before.t_h,before[col]*scale,c='gray',s=15,alpha=.55,label='Anterior al inicio')
            ax.scatter(after.t_h,after[col]*scale,c='black',s=20,zorder=6,label='Validación posterior')
            measured=subset[subset['sample'].eq(sample)].iloc[0]
            init_value=(audit.G+audit.F) if ch=='GF' else audit[ch]
            proxy=ch in ('G','F') and not audit.GF_direct
            ax.scatter([start],[init_value*scale],marker='^' if proxy else 'D',
                c='#ef6c00' if proxy else '#00897b',s=38,zorder=7,
                label='Reparto supuesto' if proxy else 'Medición inicial')
            for theta_name in THETA_SETS:
                ref=BASELINES[(lab,theta_name)]['frame']; cur=ALL_CURVES[(lab,theta_name,sample,'central')]
                color=THETA_COLORS[theta_name]
                ax.plot(ref.index,ref[ch]*scale,color=color,ls='--',alpha=.3)
                ax.plot(cur.index,cur[ch]*scale,color=color,lw=1.6,label=theta_name)
            if ch=='N':
                nut=NUT_SAMPLES[NUT_SAMPLES.lab.eq(lab)].iloc[0]
                ax.errorbar([nut.plot_t_h],[nut.N_kg_m3*scale],
                    xerr=[[nut.plot_t_h-nut.time_lower_h],[nut.time_upper_h-nut.plot_t_h]],
                    fmt='*',color='#8e24aa',ms=9,capsize=3,zorder=8,label='NUT · hora provisional')
            ax.axvline(start,color='#00897b',ls=':',lw=1)
            ax.axvline(PULSE2_H[lab],color='#d81b60',ls=':',lw=1)
            ax.set(title=f'{lab} · {ch} · inicio {start:g} h',xlim=(0,170),
                xlabel='t desde inoculación [h]',ylabel='mg N/L' if ch=='N' else 'g/L')
    handles,labels=axes[0,0].get_legend_handles_labels()
    handles2,labels2=axes[0,2].get_legend_handles_labels()
    unique=dict(zip(labels+labels2,handles+handles2))
    fig.legend(unique.values(),unique.keys(),loc='outside upper center',ncol=7,fontsize=8)
    save_figure(fig,f'03_chemistry_m{sample}_full_horizon')

## 8. Sensibilidad a cantidades no conocidas
Se perturba **una hipótesis cada vez**, sin ajustar ninguna a la química futura. La envolvente recoge X, Cd, O₂ y, para m7, el reparto G/F. No es una distribución posterior ni una banda de confianza. Las curvas de X reducido/aumentado ayudan a interpretar si el agotamiento de N depende de la biomasa inicial elegida.

In [ ]:
fig,axes=plt.subplots(3,3,figsize=(14,10),layout='constrained')
for i,lab in enumerate(LABS):
    sample='7'; theta_name='theta_full'
    central=ALL_CURVES[(lab,theta_name,sample,'central')]
    family=[c for (ll,th,ss,nn),c in ALL_CURVES.items() if (ll,th,ss)==(lab,theta_name,sample)]
    for j,ch in enumerate(('N','GF','CO2')):
        ax=axes[i,j]; column='predicted' if ch=='CO2' else ch; scale=1000. if ch=='N' else 1.
        array=np.vstack([c[column].to_numpy()*scale for c in family])
        ax.fill_between(central.index,array.min(axis=0),array.max(axis=0),alpha=.18,color='#2166ac',
                        label='Envolvente de escenarios; no IC')
        for name,color,ls in [('central','#2166ac','-'),('X_x0.5','#00897b','--'),('X_x2','#d6604d','--')]:
            cur=ALL_CURVES[(lab,theta_name,sample,name)]
            ax.plot(cur.index,cur[column]*scale,color=color,ls=ls,label=name)
        if ch=='CO2':
            obs=co2_hourly.query('lab==@lab and co2_valid and t_h>46')
            ax.scatter(obs.t_h,obs.co2_observed_g_l_h,c='black',s=6,label='CO₂ reservado')
        else:
            col=diagnostic.CHANNELS[ch]
            obs=chem.loc[chem.lab.eq(lab)&~chem.is_NUT&(chem.t_h>46)&chem[col].notna()]
            ax.scatter(obs.t_h,obs[col]*scale,c='black',s=20,label='Química reservada')
        ax.axvline(PULSE2_H[lab],color='#d81b60',ls=':')
        ax.set(title=f'{lab} · {ch} · m7 · theta_full',xlim=(40,170),xlabel='t [h]',
            ylabel='mg N/L' if ch=='N' else ('g CO₂/L/h' if ch=='CO2' else 'g/L'))
axes[0,0].legend(fontsize=7)
save_figure(fig,'04_initial_state_sensitivity')

## 9. Métricas sobre las mismas observaciones futuras
Cada comparación referencia/reinicio usa exactamente las mismas muestras posteriores al inicio y los mismos bins válidos de gas. Las métricas pre/post pulso son desgloses: **ninguna simulación se corta en el pulso**. Los inicios distintos tienen soportes distintos. Las métricas de G/F en m7 están condicionadas al reparto supuesto.

Las NUT quedan fuera del RMSE por su hora provisional. Se informa por separado el rango de N predicho sobre el intervalo de hora posible, únicamente cuando la muestra es posterior al inicio. Como NUT corresponde a la muestra posterior a nutrición, en el extremo del pulso se utiliza N(tₚ⁺)=N(tₚ⁻)+dosis. No se interpola una rampa artificial entre el valor pre-pulso y el primer punto posterior.

In [ ]:
central=METRICS.query('scenario in ["central","referencia_desde_m1"] and window=="posterior_completo"')
index=['lab','theta','sample','channel']
paired=central.pivot(index=index,columns='scenario',values=['RMSE','n'])
assert (paired['n']['central']==paired['n']['referencia_desde_m1']).all()
COMPARISON=paired['RMSE'].reset_index()
COMPARISON['n']=paired['n']['central'].to_numpy().astype(int)
COMPARISON['RMSE_ratio_restart_reference']=COMPARISON.central/COMPARISON.referencia_desde_m1
display(COMPARISON.round(5))
NUT_ROWS=[]
def nitrogen_after_action(curve,targets,schedule):
    tt=curve.index.to_numpy(float); nn=curve.N.to_numpy(float)
    targets=np.asarray(targets,float)
    result=np.interp(targets,tt,nn)
    for event,dose in schedule:
        hit=np.flatnonzero(np.isclose(tt,event,rtol=0,atol=1e-9))
        if not len(hit): continue
        i=int(hit[0]); post=float(nn[i]+dose)
        if i+1<len(tt):
            keep=(targets>=event)&(targets<=tt[i+1])
            result[keep]=post+(nn[i+1]-post)*(targets[keep]-event)/(tt[i+1]-event)
        else:
            result[np.isclose(targets,event,rtol=0,atol=1e-9)]=post
    return result
for (lab,theta_name,sample,name),curve in ALL_CURVES.items():
    if name!='central': continue
    nut=NUT_SAMPLES[NUT_SAMPLES.lab.eq(lab)].iloc[0]
    if nut.time_lower_h <= curve.index[0]: continue
    support=np.unique(np.r_[nut.time_lower_h,nut.time_upper_h,
        curve.index[(curve.index>=nut.time_lower_h)&(curve.index<=nut.time_upper_h)]])
    prediction=nitrogen_after_action(curve,support,recent_batches[lab].pulses['N'])*1000.
    NUT_ROWS.append(dict(lab=lab,theta=theta_name,sample=sample,
        observed_mg_l=nut.YAN_mg_l,time_lower_h=nut.time_lower_h,time_upper_h=nut.time_upper_h,
        predicted_min_mg_l=float(prediction.min()),predicted_max_mg_l=float(prediction.max()),
        time_provisional=nut.time_provisional))
NUT_VALIDATION=pd.DataFrame(NUT_ROWS)
display(NUT_VALIDATION.round(3))
DEPLETION=[]
for (lab,theta_name,sample,name),curve in ALL_CURVES.items():
    if name not in ('central','X_x0.5','X_x2','X_propagada'): continue
    first=curve.loc[(curve.index>=curve.index[0]) & (curve.N<=.005)]
    DEPLETION.append(dict(lab=lab,theta=theta_name,sample=sample,scenario=name,
        N_start_mg_l=curve.N.iloc[0]*1000.,
        first_predicted_N_le_5_h=float(first.index[0]) if len(first) else np.nan,
        horizon_end_h=float(curve.index[-1])))
DEPLETION_TABLE=pd.DataFrame(DEPLETION)
display(DEPLETION_TABLE.round(3))

## 10. Memoria contrafactual posterior al pulso
La inicialización química de m10 es completa, pero su producción gaseosa todavía depende de la historia contrafactual no medida. Se compara conservar el contraste nominal con borrarlo al inicio (igualar ambos estados contrafactuales), manteniendo rampa y transitorio en su reloj físico. Esto es sensibilidad estructural del reinicio, no un ajuste ni una segunda dosis. Para LAB020, todavía pre-pulso, ambos deben coincidir.

In [ ]:
HISTORY_ROWS=[]
for theta_name,th in THETA_SETS.items():
    for lab in LABS:
        ref=BASELINES[(lab,theta_name)]; start=70.5
        erased={**ref,'core0':ref['core'].copy()}
        center=ALL_CURVES[(lab,theta_name,'10','central')]
        alt=diagnostic.restart(recent_batches[lab],th,erased,start,STARTS[(lab,'10')],
            float(center.Cd.iloc[0]),float(center.O2.iloc[0]),CO2_PARAMS)
        delta=float(np.max(np.abs(alt.predicted-center.predicted)))
        if PULSE2_H[lab]>start: assert delta<1e-10
        HISTORY_ROWS.append(dict(lab=lab,theta=theta_name,past_pulse=PULSE2_H[lab]<start,
            max_CO2_change_if_history_erased_g_l_h=delta,
            past_events=repr(center.attrs['past_pulses']),future_events=repr(center.attrs['future_pulses'])))
HISTORY_SENSITIVITY=pd.DataFrame(HISTORY_ROWS)
display(HISTORY_SENSITIVITY)

## 11. Lectura de resultados y alcance
Una reducción de error después del reinicio demuestra que la información incorporada sobre el estado ayuda a la predicción; no demuestra observabilidad ni valida un EKF. Si N vuelve a caer demasiado pronto, el problema sigue presente después del arranque. Si mejora sólo con una elección particular de X, la conclusión depende de biomasa incierta. Una buena curva de gas no valida por sí sola N, G, F o E.

Las figuras y tablas siguientes son una evaluación retrospectiva ya explorada, no un holdout nuevo. Las mediciones futuras de química no se usan para elegir estados, escenarios ni parámetros. Las entradas futuras de temperatura y la convención grid_max hacen que tampoco sea todavía una prueba de despliegue online. Esta prueba **no exige muestras nuevas**; resolver una inicialización automática sigue siendo una etapa posterior.

In [ ]:
lines=['### Resultado calculado: reinicio central frente a referencia',
       'Ratios RMSE <1 indican menor error sobre exactamente las mismas observaciones futuras.']
for sample in SAMPLES:
    part=COMPARISON.query('sample==@sample and theta=="theta_full" and channel in ["N","GF","E","CO2"]')
    lines.append(f'**Inicio m{sample}: {46 if sample=="7" else 70.5} h.**')
    for lab in LABS:
        group=part[part.lab.eq(lab)]
        detail='; '.join(f'{r.channel}: {r.RMSE_ratio_restart_reference:.2f} (n={r.n})' for r in group.itertuples())
        lines.append(f'- {lab}: {detail}.')
lines.append('La muestra m7 tiene reparto G/F supuesto. m10 tiene pocos puntos químicos posteriores; '
    'no se interpreta una mejora con n=1 como validación general. Revisar sensibilidad X/Cd/O₂ y memoria nutricional antes de concluir.')
display(Markdown('\n\n'.join(lines)))

tables=dict(initialization=INITIALIZATION,gas_initialization=GAS_INITIALIZATION,
    parameters_theta=THETA_TABLE,numerical_controls=CONTROL_TABLE,checks=CHECKS,
    metrics=METRICS,paired_comparison=COMPARISON,nut_reserved_validation=NUT_VALIDATION,
    nitrogen_depletion=DEPLETION_TABLE,history_sensitivity=HISTORY_SENSITIVITY,
    mask_audit=mask_audit)
for name,table in tables.items(): table.to_csv(OUT/(name+'.csv'),index=False)
NUT_SAMPLES.to_csv(OUT/'nut_sample_audit.csv',index=False)
trajectory_parts=[]; compressed={}; array_manifest=[]
for i,(key,cur) in enumerate(ALL_CURVES.items()):
    lab,theta_name,sample,name=key
    if name=='central':
        part=cur.copy().reset_index(names='t_h')
        part['tau_h']=part.t_h-part.t_h.iloc[0]
        part['lab'],part['theta'],part['sample']=lab,theta_name,sample
        trajectory_parts.append(part)
    columns=list(cur.columns)
    compressed[f'case_{i}']=np.column_stack([cur.index.to_numpy(),cur.to_numpy()])
    array_manifest.append(dict(array=f'case_{i}',key=list(key),columns=['t_h']+columns))
pd.concat(trajectory_parts,ignore_index=True).to_csv(OUT/'central_trajectories.csv',index=False)
np.savez_compressed(OUT/'scenario_trajectories.npz',**compressed)
for (lab,theta_name),ref in BASELINES.items():
    ref['frame'].to_csv(OUT/f'baseline_{theta_name}_{lab}.csv',index_label='t_h')
HASH_AFTER={str(p.relative_to(ROOT)):sha256(p) for p in PROTECTED}
assert HASH_BEFORE==HASH_AFTER,'Un input protegido cambió durante el diagnóstico'
assert not CHECKS.counterfactual_clipped.ne('{}').any(), 'Revisar inicialización contrafactual negativa'
manifest=dict(created_at=datetime.now().astimezone().isoformat(),python=sys.executable,
    config=diagnostic.CONFIG,theta=THETA_SETS,co2_parameters=CO2_PARAMS,
    input_sha256=HASH_BEFORE,arrays=array_manifest,optimizer_calls=0,kalman_updates=0,
    starts_h=[46.,70.5],simulation_end_h={lab:float(recent_batches[lab].time[-1]) for lab in LABS},
    scientific_scope='retrospective measured-chemistry restart; fixed historical parameters; full horizon; not online validation',
    gate='nominal history retained per theta; not restarted',
    counterfactual='carry nominal contrast at restart; report history-erasure sensitivity',
    sensitivity='one-at-a-time exploratory scenarios; not confidence intervals',
    biomass='uncertain proxy; excluded from validation',
    dissolved_CO2='past-flow inversion with frozen observation parameters; not a dissolved measurement')
(OUT/'manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
print('Integridad OK:',len(PROTECTED),'inputs protegidos; curvas hasta',manifest['simulation_end_h'])
print('Sin ajuste de parámetros, sin EKF, sin modificar datos originales ni notebooks previos.')
print('Resultados:',OUT)

## 12. Conclusiones de esta ejecución

**El reinicio desde química medida ayuda a algunas predicciones, pero no corrige por sí solo el nitrógeno ni asegura una buena salida gaseosa.** Las conclusiones siguientes corresponden a los parámetros congelados y a los escenarios exploratorios de este notebook; no equivalen a validar el EKF.

**Química y gas cuentan historias distintas.** En el reinicio m7 (46 h), theta_full reduce el RMSE posterior de GF y etanol respecto de la referencia desde m1 en los tres labs. Sin embargo, su RMSE de CO₂ aumenta aproximadamente 23%, 5% y 12% en LAB019, LAB020 y LAB021. Los defaults dan menor RMSE de gas que theta_full en esos tres reinicios, pero mayores errores de GF y etanol. No debemos seleccionar theta usando únicamente CO₂ ni atribuir todo el desajuste al reloj inicial. Estas comparaciones utilizan el mismo soporte posterior a las 46 h; no comparan una ventana corta con una fermentación completa.

**N vuelve a agotarse demasiado pronto en el escenario central.** Desde N medido a las 46 h, theta_full predice el primer cruce N≤5 mg/L a las 53,25 h, 60,25 h y 47 h para LAB019, LAB020 y LAB021. Ambos theta presentan el mismo problema cualitativo. Las simulaciones continúan después de esos cruces, aplican los pulsos y llegan a las 166 h: la línea próxima a cero posterior no es un recorte. La sensibilidad de biomasa cambia los tiempos; no permite atribuir el fallo exclusivamente a qN, porque X es incierto y el estado N se está equiparando a YAN.

**Las NUT ayudan a localizar el problema.** En m7, LAB019 y LAB020 llegan al pulso con N prácticamente agotado; añadir 80 mg/L deja máximos posteriores de aproximadamente 80 mg/L, inferiores a sus NUT (99,734 y 108,495 mg/L calculados). Cambiar la hora de extracción dentro del intervalo provisional no elimina esa diferencia. En cambio, reinicializar LAB020 a las 70,5 h conserva nitrógeno medido justo antes del pulso y produce un rango NUT de aproximadamente 102,25–109,82 mg/L con theta_full, compatible con 108,495 mg/L. Aun así, N cae nuevamente demasiado rápido después: acertar una NUT no valida la dinámica posterior.

**m10 comprueba un reinicio con G/F disponibles, pero tiene menos validación química posterior.** Sólo quedan dos muestras de N/E y, en algunos canales de azúcar, una única muestra válida. Además, LAB019 y LAB021 ya recibieron nutrición: borrar la memoria contrafactual cambia materialmente el gas (hasta aproximadamente 0,166 y 0,076 g/L/h con theta_full). Por ello su predicción gaseosa depende también de una historia auxiliar no medida. En LAB020, antes del pulso, ese contraste de historias es exactamente cero.

**Decisión para el trabajo:** el corrimiento/reinicio es un diagnóstico útil, pero todavía no una solución de inicialización automática. Antes de confiar en una corrección por EKF, conviene revisar conjuntamente biomasa efectiva, captación de N y correspondencia YAN–estado N, y comprobar la capa de CO₂ contra la química. No se ha añadido latencia, cambiado la cinética ni reajustado parámetros en esta prueba. Las bandas son sensibilidad a hipótesis, no intervalos de confianza.
